# Duka Electronics Sales Exploration

## Executive Summary

- Nairobi was the top-performing branch, generating KES 26,106,100 from 376 orders.
- Laptops generated the highest product revenue at KES 41,535,000, while Tablets led unit sales with 691 units.
- Weekdays generated KES 51,856,200 from 723 orders, while weekends generated KES 19,936,300 from 277 orders and had a slightly higher average order revenue of KES 71,972.20.
- April was the strongest revenue month at KES 13,518,300, which was 23.98% higher than June, the weakest month at KES 10,903,400.
- The Premium Online segment from Nairobi and Mombasa contained 83 orders and generated KES 10,218,000 in total revenue.

## Task 1 — First-Five-Minutes Inspection

The dataset is loaded with the `order_date` column parsed as a date. The initial inspection checks the dataset shape, first and last records, data types, missing values, descriptive statistics, date range, distinct values, and key categorical distributions before analysis.

In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

print("Dataset shape:", sales.shape)
sales.head()

Dataset shape: (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [2]:
print("Last five rows:")
display(sales.tail())

print("\nColumn data types:")
print(sales.dtypes)

print("\nDataset information:")
sales.info()

print("\nMissing values:")
print(sales.isna().sum())

Last five rows:


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200



Column data types:
order_id               int64
order_date    datetime64[us]
branch                   str
product                  str
quantity               int64
channel                  str
unit_price             int64
dtype: object

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB

Missing values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64


In [3]:
print("Descriptive statistics:")
display(sales.describe())

print("\nDate range:")
print("Earliest order date:", sales["order_date"].min())
print("Latest order date:", sales["order_date"].max())

print("\nDistinct values per column:")
print(sales.nunique())

print("\nBranch counts:")
print(sales["branch"].value_counts())

print("\nChannel counts:")
print(sales["channel"].value_counts())

print("\nProduct counts:")
print(sales["product"].value_counts())

Descriptive statistics:


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546



Date range:
Earliest order date: 2026-01-01 00:00:00
Latest order date: 2026-06-29 00:00:00

Distinct values per column:
order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64

Branch counts:
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64

Channel counts:
channel
In-store       579
Online         320
Phone order    101
Name: count, dtype: int64

Product counts:
product
Laptop        220
Tablet        215
Phone         194
Headphones    187
Charger       184
Name: count, dtype: int64


### First-Five-Minutes Inspection Findings

- **Shape:** The dataset contains 1,000 orders and 7 original columns.
- **Data types:** `order_id`, `quantity`, and `unit_price` are integer columns; `order_date` is correctly parsed as a datetime column; and `branch`, `product`, and `channel` are text columns.
- **Missing values:** No missing values were found; all seven columns contain 1,000 non-null records.
- **Date range:** Orders cover January 1, 2026 through June 29, 2026, representing the required January–June 2026 review period.
- **Key categories:** The dataset contains 5 branches, 5 products, and 3 sales channels. Nairobi has the largest number of orders with 376.
- **Numeric ranges:** Order quantities range from 1 to 5 units, while unit prices range from KES 1,200 to KES 65,000.

## Task 2 — Derived Columns

Five derived columns are created to support the sales analysis: `revenue`, `month`, `weekday`, `is_weekend`, and `price_band`. Premium products have a unit price of at least KES 28,000; all other products are classified as Value.

In [4]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]

sales["month"] = sales["order_date"].dt.month_name()

sales["weekday"] = sales["order_date"].dt.day_name()

sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5

sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


## Task 3 — Branch Rankings

In [5]:
branch_revenue = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

branch_orders = (
    sales.groupby("branch")["order_id"]
    .count()
    .sort_values(ascending=False)
)

print("Branch ranking by total revenue:")
print(branch_revenue)

print("\nBranch ranking by number of orders:")
print(branch_orders)

Branch ranking by total revenue:
branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200
Name: revenue, dtype: int64

Branch ranking by number of orders:
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: order_id, dtype: int64


### Answer

The branch rankings by total revenue and number of orders are the same. Nairobi ranks first with KES 26,106,100 in revenue and 376 orders, followed by Mombasa with KES 14,003,700 and 186 orders. Nakuru ranks third with KES 11,970,400 and 162 orders, Kisumu fourth with KES 10,868,100 and 159 orders, and Eldoret fifth with KES 8,844,200 and 117 orders. The rankings are the same because branches with more orders also generated more total revenue in this dataset, so differences in product mix and order value were not large enough to change the ordering.

## Task 4 — Top Three Products by Revenue and Units Sold

In [6]:
product_revenue = (
    sales.groupby("product")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

product_units = (
    sales.groupby("product")["quantity"]
    .sum()
    .sort_values(ascending=False)
)

print("Top three products by revenue:")
print(product_revenue.head(3))

print("\nTop three products by units sold:")
print(product_units.head(3))

Top three products by revenue:
product
Laptop    41535000
Tablet    19348000
Phone      8880000
Name: revenue, dtype: int64

Top three products by units sold:
product
Tablet    691
Laptop    639
Phone     592
Name: quantity, dtype: int64


### Answer

The top three products by revenue are Laptop (KES 41,535,000), Tablet (KES 19,348,000), and Phone (KES 8,880,000). By units sold, the top three are Tablet (691 units), Laptop (639 units), and Phone (592 units). The same three products appear in both lists, but their rankings differ: Laptop ranks first in revenue while Tablet ranks first in units sold. This difference occurs because Laptop has a higher unit price than Tablet, allowing it to generate more revenue despite selling fewer units.

## Task 5 — Weekend vs Weekday Orders

In [7]:
weekend_comparison = (
    sales.groupby("is_weekend")
    .agg(
        number_of_orders=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        average_order_revenue=("revenue", "mean")
    )
)

weekend_comparison.index = weekend_comparison.index.map({
    False: "Weekday",
    True: "Weekend"
})

print("Weekend vs Weekday comparison:")
display(weekend_comparison)

Weekend vs Weekday comparison:


,number_of_orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


### Weekend vs Weekday Findings

Weekdays recorded 723 orders and generated KES 51,856,200 in total revenue, compared with 277 weekend orders generating KES 19,936,300. Although weekdays account for substantially more orders and total revenue, the average revenue per order is slightly higher on weekends at approximately KES 71,972.20, compared with KES 71,723.65 on weekdays.

## Task 6 — Monthly Revenue Trend

In [8]:
monthly_revenue = (
    sales.groupby("month")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

best_month = monthly_revenue.idxmax()
best_revenue = monthly_revenue.max()

worst_month = monthly_revenue.idxmin()
worst_revenue = monthly_revenue.min()

percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

print("Revenue by month:")
print(monthly_revenue)

print(f"\nBest month: {best_month} - KES {best_revenue:,}")
print(f"Worst month: {worst_month} - KES {worst_revenue:,}")
print(f"Percentage difference: {percentage_difference:.2f}%")

Revenue by month:
month
April       13518300
March       12396800
January     11786600
February    11772000
May         11415400
June        10903400
Name: revenue, dtype: int64

Best month: April - KES 13,518,300
Worst month: June - KES 10,903,400
Percentage difference: 23.98%


### Answer

April was the best-performing month, generating KES 13,518,300 in revenue, while June was the worst-performing month with KES 10,903,400. April's revenue was approximately 23.98% higher than June's revenue, showing a noticeable difference between the strongest and weakest months in the January–June 2026 period.

## Task 7 — Premium Online Orders from Nairobi and Mombasa

In [9]:
premium_online = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

premium_online_count = len(premium_online)
premium_online_revenue = premium_online["revenue"].sum()

print("Number of Premium Online orders from Nairobi and Mombasa:",
      premium_online_count)

print(f"Total revenue: KES {premium_online_revenue:,}")

premium_online.head()

Number of Premium Online orders from Nairobi and Mombasa: 83
Total revenue: KES 10,218,000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


In [10]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("Export completed successfully.")

Export completed successfully.


### Answer

There are 83 Premium Online orders from Nairobi and Mombasa, generating a total revenue of KES 10,218,000. These filtered orders were exported to `outputs/premium_online_coast_capital.csv` with the DataFrame index excluded.

In [11]:
check_export = pd.read_csv(
    "outputs/premium_online_coast_capital.csv"
)

print("Exported rows:", len(check_export))
print("Branches:", check_export["branch"].unique())
print("Channels:", check_export["channel"].unique())
print("Price bands:", check_export["price_band"].unique())
print(f"Total revenue: KES {check_export['revenue'].sum():,}")

Exported rows: 83
Branches: <StringArray>
['Nairobi', 'Mombasa']
Length: 2, dtype: str
Channels: <StringArray>
['Online']
Length: 1, dtype: str
Price bands: <StringArray>
['Premium']
Length: 1, dtype: str
Total revenue: KES 10,218,000


## Task 8 — Excel Validation

In [12]:
excel_branch_totals = {
    "Nairobi": 26106100,
    "Mombasa": 14003700,
    "Nakuru": 11970400,
    "Kisumu": 10868100,
    "Eldoret": 8844200
}

excel_total = sum(excel_branch_totals.values())
pandas_total = sales["revenue"].sum()

print(f"Excel PivotTable Grand Total: KES {excel_total:,}")
print(f"Pandas Total Revenue: KES {pandas_total:,}")
print("Totals match:", excel_total == pandas_total)

Excel PivotTable Grand Total: KES 71,792,500
Pandas Total Revenue: KES 71,792,500
Totals match: True


### Excel Validation Result

The Excel workbook reproduces the branch revenue analysis using SUMIFS, adds branch regions using XLOOKUP, and includes a PivotTable of revenue by branch and channel. The Excel PivotTable Grand Total is KES 71,792,500, which exactly matches the pandas total revenue of KES 71,792,500.